# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`

This notebook provides a step-by-step demonstration for loading and exploring the FAIR² dataset using the [mlcroissant](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Install mlcroissant if not already available
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant metadata schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Instantiate the Dataset
dataset = mlc.Dataset(croissant_url)

# Accessing metadata as an object: print summary
meta_obj = dataset.metadata
print(f"Dataset Name: {meta_obj.name}\n\nDescription: {meta_obj.description}\n")

## 2. Data Overview

Review available record sets (`@id`s), the included fields, and their corresponding `@id`s.

Below, we enumerate all record sets, and for each, print the fields and columns by their `@id`. This step will equip you with the identifiers required for extraction and further processing.

In [ ]:
# Obtain all record sets inside the dataset
record_sets = list(dataset.record_sets())  # Returns MlcRecordSet objects

print(f"Number of record sets: {len(record_sets)}\n")
for rs in record_sets:
    print(f"Record Set: {rs.id}")
    if hasattr(rs, 'fields') and rs.fields:
        print(" Fields:")
        for field in rs.fields:
            if hasattr(field, 'id'):
                print(f"   - {field.id}")
    if hasattr(rs, 'columns') and rs.columns:
        print(" Columns:")
        for col in rs.columns:
            if hasattr(col, 'id'):
                print(f"   - {col.id}")
    print("---")

## 3. Data Extraction

Load data from all available record sets into DataFrames for analysis. All entities are referenced using their `@id` as shown in the overview above. This step loads all structured content and shows the first record set's data preview.

In [ ]:
# Collect all record_set @id's
record_set_ids = [rs.id for rs in dataset.record_sets()]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)

# Show info for the first non-empty record set
for rsid, df in dataframes.items():
    print(f"Record Set '@id': {rsid}")
    print(f"Columns: {df.columns.tolist()}")
    display(df.head())
    break  # Preview the first one only

## 4. Exploratory Data Analysis (EDA)

Apply common EDA operations on one selected record set. To demonstrate, we:
- Select a numeric field (by `@id`)
- Filter records on a threshold,
- Normalize the numeric values,
- Optionally group data by a categorical field (`@id`).

> **Edit the cell below if you want to try this with a different record set, numeric field, or group field (all referenced by their `@id`).

In [ ]:
# Demo: Pick the first record set with numeric columns
chosen_rs_id = None
chosen_numeric_field = None
chosen_group_field = None
for rsid, df in dataframes.items():
    # Attempt to find a numeric-looking column
    numeric_candidates = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    if numeric_candidates:
        chosen_rs_id = rsid
        chosen_numeric_field = numeric_candidates[0]
        # Choose another column for group-by (not the numeric one)
        non_numeric = [col for col in df.columns if col != chosen_numeric_field and df[col].nunique() < 20]
        if non_numeric:
            chosen_group_field = non_numeric[0]
        break

if chosen_rs_id and chosen_numeric_field:
    print(f"Using record set: {chosen_rs_id}")
    print(f"Numeric field: {chosen_numeric_field}")
    if chosen_group_field:
        print(f"Group by: {chosen_group_field}")
    df = dataframes[chosen_rs_id].copy()

    # Filter by threshold
    threshold = df[chosen_numeric_field].mean()
    filtered_df = df[df[chosen_numeric_field] > threshold]
    print(f"Filtered rows with {chosen_numeric_field} > {threshold:.2f} (mean): {len(filtered_df)}\n")
    
    # Normalize
    norm_col = f"{chosen_numeric_field}_normalized"
    filtered_df[norm_col] = (filtered_df[chosen_numeric_field] - filtered_df[chosen_numeric_field].mean()) / filtered_df[chosen_numeric_field].std()
    print("Normalized values:")
    print(filtered_df[[chosen_numeric_field, norm_col]].head())

    # Group by a field if available
    if chosen_group_field:
        grouped = filtered_df.groupby(chosen_group_field)[chosen_numeric_field].mean().reset_index()
        print(f"\nMean {chosen_numeric_field} by {chosen_group_field}:")
        print(grouped.head())
else:
    print("No suitable numeric field found for demonstration.")

## 5. Visualization

Visualize the distribution of the selected numeric field, and the relationship between the group and mean field value if available.

In [ ]:
import matplotlib.pyplot as plt

if chosen_rs_id and chosen_numeric_field:
    plt.figure(figsize=(7,4))
    df = dataframes[chosen_rs_id]
    df[chosen_numeric_field].hist(bins=20, color='#4287f5')
    plt.xlabel(chosen_numeric_field)
    plt.ylabel("Frequency")
    plt.title(f"Histogram of {chosen_numeric_field}")
    plt.show()

    if chosen_group_field:
        # Bar plot of means by group
        plt.figure(figsize=(7,4))
        grouped = df.groupby(chosen_group_field)[chosen_numeric_field].mean().reset_index()
        plt.bar(grouped[chosen_group_field].astype(str), grouped[chosen_numeric_field], color='#23bfae')
        plt.xlabel(chosen_group_field)
        plt.ylabel(f"Mean {chosen_numeric_field}")
        plt.title(f"Mean {chosen_numeric_field} by {chosen_group_field}")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

## 6. Conclusion

In this notebook, we accessed the FAIR² dataset using `mlcroissant`, printed the available record sets and their fields (referenced by their `@id`s), loaded the data into Pandas DataFrames, and performed some basic EDA and visualization.

To go further, reference the `@id` fields from the Data Overview step when extracting or processing specific data elements. You can extend this workflow by applying custom analyses, merging with other data, or automating data extraction across multiple Croissant-conformant packages.

For more information, refer to the [mlcroissant documentation](https://mlcroissant.readthedocs.io/) or the dataset's FAIR metadata for detailed structure.